# LangGraph Fundamentals — Build Pro-Code Agents on SAP BTP

This notebook is a structured learning path for LangGraph, culminating in how to run agents against the **SAP Generative AI Hub** on SAP BTP.

## Roadmap

| # | Section | Key Concept |
|---|---------|-------------|
| 1 | Setup | Install packages, set keys |
| 2 | Core Building Blocks | State · Nodes · Edges |
| 3 | First Graph | Simple linear pipeline |
| 4 | Conditional Routing | Branching with router functions |
| 5 | Tools & ToolNode | Giving agents superpowers |
| 6 | ReAct Agent Loop | The standard agentic pattern |
| 7 | Persistence & Memory | MemorySaver · thread_id |
| 8 | Human-in-the-Loop | Interrupt · approve · resume |
| 9 | SAP BTP Integration | Generative AI Hub + LangGraph |

---

## Section 1 — Setup

### Installation

```bash
# Core LangGraph packages
pip install langgraph langchain-openai langchain-core

# SAP BTP — Generative AI Hub
# sap-ai-sdk-gen is the current package (renamed from generative-ai-hub-sdk).
# Import paths (gen_ai_hub.*) are IDENTICAL — existing code works unchanged.
pip install "sap-ai-sdk-gen[all]"

# Nice extras for development
pip install langgraph-checkpoint-sqlite  # persistent memory via SQLite
pip install grandalf                      # graph visualisation
```

If you are using **uv** (which this project uses):
```bash
uv add langgraph langchain-openai langchain-core "sap-ai-sdk-gen[all]"
```

In [1]:
# Verify installed versions using importlib.metadata (works for all packages)
from importlib.metadata import version, PackageNotFoundError

packages = {
    "langgraph": "langgraph",
    "langchain_core": "langchain-core",
    "langchain_openai": "langchain-openai",
    "sap-ai-sdk-gen": "sap-ai-sdk-gen",      # SAP BTP SDK (Section 9)
}

for display, pkg_name in packages.items():
    try:
        print(f"{display:30s} {version(pkg_name)}")
    except PackageNotFoundError:
        print(f"{display:30s} NOT INSTALLED — run: pip install {pkg_name}")

langgraph                      1.0.9
langchain_core                 1.2.14
langchain_openai               1.1.10
sap-ai-sdk-gen                 NOT INSTALLED — run: pip install sap-ai-sdk-gen


### API Keys (non-BTP path)

If you want to run the generic examples without SAP BTP, set a standard OpenAI key.
For BTP, see **Section 9** instead.

In [2]:
import os
from dotenv import load_dotenv

# Load variables from .env in the project root
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not found. "
        "Add it to your .env file: OPENAI_API_KEY=sk-..."
    )

print("OPENAI_API_KEY loaded.")

OPENAI_API_KEY loaded.


---
## Section 2 — Core Building Blocks

LangGraph has three primitives:

```
┌───────────────────────────────────────────────────────┐
│  STATE  ──►  NODE  ──►  EDGE  ──►  NODE  ──►  END     │
│               │                     ▲                 │
│               └─────────────────────┘  (loop/cycle)   │
└───────────────────────────────────────────────────────┘
```

### 2.1 State

State is a **typed dictionary** that flows through every node.  
Nodes read from it and return **partial updates** — LangGraph merges them automatically.

The `Annotated` + reducer pattern is key for lists (messages, history, etc.):

In [21]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages  # built-in reducer: appends messages


# --- Minimal state for a chat agent ---
class ChatState(TypedDict):
    # add_messages reducer: each node returns new messages; LangGraph *appends* them.
    # Without the reducer, returning a list would *replace* the field entirely.
    messages: Annotated[List[BaseMessage], add_messages]


# --- Richer state for a research agent ---
class ResearchState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    topic: str           # plain field — last write wins
    sources: List[str]   # plain list — last write wins (replace, not append)
    draft: str
    approved: bool

print("State schemas defined.")

State schemas defined.


### 2.2 Nodes

A node is a **plain Python function** (or async function, or any callable).

- Input: the current `State`
- Output: a **dict** containing only the fields you want to update

LangGraph merges your dict into the state. You never need to copy the whole state.

In [22]:
from langchain_core.messages import AIMessage


def greeter_node(state: ChatState) -> dict:
    """Returns a greeting based on the last user message."""
    last_msg = state["messages"][-1].content
    reply = AIMessage(content=f"Hello! You said: '{last_msg}'")
    # Only return what we want to update — LangGraph merges this
    return {"messages": [reply]}


# Async nodes are supported too:
async def async_node(state: ChatState) -> dict:
    import asyncio
    await asyncio.sleep(0)  # simulate async I/O
    return {"messages": [AIMessage(content="async response")]}

print("Node functions defined.")

Node functions defined.


### 2.3 Edges

| Edge type | When to use |
|-----------|-------------|
| `add_edge(A, B)` | Always go from A → B |
| `add_conditional_edges(A, router_fn)` | Decide at runtime which node to go to |
| `START` / `END` | Built-in sentinels for entry and exit |

A **router function** receives the state and returns a string (node name or `END`).

---
## Section 3 — Your First Graph (Linear Pipeline)

We build a simple 3-step pipeline:
```
START → preprocess → generate → postprocess → END
```

In [23]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


# ── State ──────────────────────────────────────────────────────────────────
class PipelineState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    processed_input: str


# ── Nodes ──────────────────────────────────────────────────────────────────
def preprocess(state: PipelineState) -> dict:
    raw = state["messages"][-1].content
    cleaned = raw.strip().lower()
    print(f"[preprocess] '{raw}' → '{cleaned}'")
    return {"processed_input": cleaned}


def generate(state: PipelineState) -> dict:
    text = state["processed_input"]
    # Placeholder: in real code, call your LLM here
    response = f"Echo (mocked LLM): {text}"
    print(f"[generate] → '{response}'")
    return {"messages": [AIMessage(content=response)]}


def postprocess(state: PipelineState) -> dict:
    last = state["messages"][-1].content
    final = last.upper()  # simulate formatting
    print(f"[postprocess] → '{final}'")
    return {"messages": [AIMessage(content=final)]}


# ── Assemble the graph ─────────────────────────────────────────────────────
builder = StateGraph(PipelineState)

builder.add_node("preprocess", preprocess)
builder.add_node("generate", generate)
builder.add_node("postprocess", postprocess)

builder.add_edge(START, "preprocess")
builder.add_edge("preprocess", "generate")
builder.add_edge("generate", "postprocess")
builder.add_edge("postprocess", END)

pipeline = builder.compile()

# ── Run ────────────────────────────────────────────────────────────────────
initial_state = {"messages": [HumanMessage(content="  Hello World!  ")]}
result = pipeline.invoke(initial_state)

print("\n=== Final message ===")
print(result["messages"][-1].content)

[preprocess] '  Hello World!  ' → 'hello world!'
[generate] → 'Echo (mocked LLM): hello world!'
[postprocess] → 'ECHO (MOCKED LLM): HELLO WORLD!'

=== Final message ===
ECHO (MOCKED LLM): HELLO WORLD!


In [24]:
# Visualise the graph (requires grandalf or pygraphviz)
try:
    print(pipeline.get_graph().draw_ascii())
except Exception:
    print("Install 'grandalf' for ASCII visualisation: pip install grandalf")

# Or save as PNG (requires pygraphviz):
# png = pipeline.get_graph().draw_mermaid_png()
# with open("graph.png", "wb") as f:
#     f.write(png)

 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
+------------+   
| preprocess |   
+------------+   
        *        
        *        
        *        
  +----------+   
  | generate |   
  +----------+   
        *        
        *        
        *        
+-------------+  
| postprocess |  
+-------------+  
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


---
## Section 4 — Conditional Routing

A **router function** inspects the state and returns the name of the next node.

```
START → classify ─┬─ "simple_path" ──► simple_handler ──► END
                  └─ "complex_path" ─► complex_handler ──► END
```

In [25]:
from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class RouterState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    complexity: str   # "simple" | "complex"


# ── Nodes ──────────────────────────────────────────────────────────────────
def classify(state: RouterState) -> dict:
    """Determine whether the question is simple or complex."""
    question = state["messages"][-1].content.lower()
    # Placeholder rule — replace with LLM classification in production
    complexity = "complex" if len(question.split()) > 6 else "simple"
    print(f"[classify] complexity = {complexity}")
    return {"complexity": complexity}


def simple_handler(state: RouterState) -> dict:
    print("[simple_handler] answering directly")
    return {"messages": [AIMessage(content="Quick answer: 42.")]}


def complex_handler(state: RouterState) -> dict:
    print("[complex_handler] thinking hard...")
    return {"messages": [AIMessage(content="Complex answer after deep analysis.")]}


# ── Router function ────────────────────────────────────────────────────────
# Must return a string matching a node name, or END
def route_by_complexity(state: RouterState) -> Literal["simple_handler", "complex_handler"]:
    return "simple_handler" if state["complexity"] == "simple" else "complex_handler"


# ── Build graph ────────────────────────────────────────────────────────────
builder = StateGraph(RouterState)
builder.add_node("classify", classify)
builder.add_node("simple_handler", simple_handler)
builder.add_node("complex_handler", complex_handler)

builder.add_edge(START, "classify")
# Conditional edge: after classify, call route_by_complexity to decide next node
builder.add_conditional_edges(
    "classify",
    route_by_complexity,
    # Optional explicit mapping (good for clarity):
    {"simple_handler": "simple_handler", "complex_handler": "complex_handler"}
)
builder.add_edge("simple_handler", END)
builder.add_edge("complex_handler", END)

router_graph = builder.compile()

# ── Test both paths ────────────────────────────────────────────────────────
for q in ["What is 2+2?", "Can you explain the theory of relativity in detail?"]:
    print(f"\nQ: {q}")
    res = router_graph.invoke({"messages": [HumanMessage(content=q)]})
    print(f"A: {res['messages'][-1].content}")


Q: What is 2+2?
[classify] complexity = simple
[simple_handler] answering directly
A: Quick answer: 42.

Q: Can you explain the theory of relativity in detail?
[classify] complexity = complex
[complex_handler] thinking hard...
A: Complex answer after deep analysis.


---
## Section 5 — Tools and ToolNode

Tools are the **superpowers** you give an agent — web search, database queries, calculators, APIs, etc.

Three steps:
1. Decorate a function with `@tool`
2. Bind tools to the LLM with `llm.bind_tools(tools)`
3. Use LangGraph's built-in `ToolNode` to execute tool calls

In [26]:
from langchain_core.tools import tool


# ── Define tools ───────────────────────────────────────────────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Input must be a valid Python math expression.
    Example: '2 + 2 * 3'"""
    try:
        # WARNING: eval is used for illustration only — sanitise in production!
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city. Returns a weather description."""
    # Mocked — replace with a real weather API call
    weather_db = {
        "berlin": "15°C, partly cloudy",
        "walldorf": "12°C, sunny",
        "new york": "22°C, clear skies",
    }
    return weather_db.get(city.lower(), f"No weather data for '{city}'.")


tools = [calculator, get_weather]

# Inspect the tool schema (what the LLM sees)
print(calculator.name)
print(calculator.description)
print(calculator.args_schema.schema())

calculator
Evaluate a mathematical expression. Input must be a valid Python math expression.
Example: '2 + 2 * 3'
{'description': "Evaluate a mathematical expression. Input must be a valid Python math expression.\nExample: '2 + 2 * 3'", 'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculator', 'type': 'object'}


/var/folders/_s/q_tn2szx585b2f9yp0gj2ffc0000gn/T/ipykernel_15559/3888858164.py:34: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(calculator.args_schema.schema())


In [27]:
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

# ToolNode is a pre-built node that executes tool calls found in the last AIMessage
tool_node = ToolNode(tools)

# In LangGraph 1.0+, ToolNode._func requires a Runtime object injected by the graph
# engine — it cannot be called standalone via invoke(). Wrap it in a minimal graph instead.
class _TestState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

_builder = StateGraph(_TestState)
_builder.add_node("tools", tool_node)
_builder.add_edge(START, "tools")
_builder.add_edge("tools", END)
_test_graph = _builder.compile()

fake_tool_call = AIMessage(
    content="",
    tool_calls=[{
        "name": "calculator",
        "args": {"expression": "3 ** 4 + 12"},
        "id": "call_001",
        "type": "tool_call"
    }]
)

result = _test_graph.invoke({"messages": [fake_tool_call]})
print(result["messages"][-1].content)   # → 93

93


---
## Section 6 — ReAct Agent Loop

The **ReAct pattern** (Reasoning + Acting) is the most common agentic pattern:

```
START → agent ──── has_tool_calls? ──► tool_node ──┐
                         │                          │
                    no: END               (loop back to agent)
```

The agent decides to call a tool or produce a final answer. The loop runs until no more tool calls are made.

In [29]:
# NOTE: Requires OPENAI_API_KEY or a BTP proxy (see Section 9)
# If you only have BTP access, skip to Section 9 and come back here.

from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


# ── State ──────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


# ── LLM + tool binding ─────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)   # tools defined in Section 5


# ── Nodes ──────────────────────────────────────────────────────────────────
def agent_node(state: AgentState) -> dict:
    """Call the LLM with the current conversation history."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


tool_node = ToolNode(tools)


# ── Router ─────────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """If the last message has tool calls → run tools. Otherwise → done."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "__end__"


# ── Build the ReAct graph ───────────────────────────────────────────────────
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue)
builder.add_edge("tools", "agent")   # after tools → back to agent

react_agent = builder.compile()

print("ReAct agent compiled.")
try:
    print(react_agent.get_graph().draw_ascii())
except Exception:
    pass

ReAct agent compiled.
        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [30]:
# Run the agent (requires LLM access)
response = react_agent.invoke({
    "messages": [HumanMessage(content="What is 17 * 83, and what is the weather in Berlin?")]
})

print("=== Conversation trace ===")
for msg in response["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    content = msg.content or str(getattr(msg, "tool_calls", ""))
    print(f"[{role}] {content[:120]}")

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

### Shortcut: `create_agent`

LangGraph ships a pre-built helper (now in `langchain.agents`) that builds exactly the graph above in one line:

In [32]:
from langchain.agents import create_agent

# Equivalent to the manual graph above — great for quick prototyping
quick_agent = create_agent(
    model=llm,
    tools=tools,
    # Optional system prompt:
    system_prompt="You are a helpful assistant. Use tools when appropriate."
)

result = quick_agent.invoke({"messages": [HumanMessage(content="Calculate 2**10")]})
print(result["messages"][-1].content)

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

---
## Section 7 — Persistence & Memory

By default, every `invoke()` call starts with a blank state. **Checkpointers** save and restore state between calls, enabling:
- Multi-turn conversations (the agent remembers what was said)
- Resuming after failures
- Auditing (inspect state at any point)

```
thread_id = "user-123"
   ↓
invoke(msg_1)  →  saves checkpoint  →  invoke(msg_2)  →  loads checkpoint, continues
```

In [33]:
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver stores state in RAM — good for development/testing.
# For production, use SqliteSaver or PostgresSaver.
memory = MemorySaver()

# Compile the agent WITH a checkpointer
persistent_agent = builder.compile(checkpointer=memory)
# (builder is from Section 6 above)

# Each conversation thread needs a unique config
config_thread_1 = {"configurable": {"thread_id": "conv-001"}}
config_thread_2 = {"configurable": {"thread_id": "conv-002"}}

# Turn 1 — user greets
r1 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Alex.")]},
    config=config_thread_1
)
print("Turn 1:", r1["messages"][-1].content)

# Turn 2 — agent should remember the name
r2 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config_thread_1   # same thread_id!
)
print("Turn 2:", r2["messages"][-1].content)

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [34]:
# ── Inspect stored state ────────────────────────────────────────────────────
snapshot = persistent_agent.get_state(config_thread_1)
print("Current node:", snapshot.next)         # empty = finished
print("Message count:", len(snapshot.values["messages"]))

# ── SQLite persistence (survives process restart) ───────────────────────────
# from langgraph.checkpoint.sqlite import SqliteSaver
# with SqliteSaver.from_conn_string(":memory:") as sqlite_memory:
#     persistent_agent = builder.compile(checkpointer=sqlite_memory)
#     ...

Current node: ('agent',)
Message count: 1


---
## Section 8 — Human-in-the-Loop

LangGraph can **pause execution before (or after) any node** and wait for human input. This is critical for:
- Approving destructive actions (delete, send email, execute SQL)
- Reviewing draft content
- Guided workflows requiring sign-off

```
START → agent → [PAUSE] → tools → agent → END
                    ▲
             human approves here
```

In [35]:
from langgraph.checkpoint.memory import MemorySaver

hitl_memory = MemorySaver()

# interrupt_before=["tools"] → pause BEFORE the tools node runs
hitl_agent = builder.compile(
    checkpointer=hitl_memory,
    interrupt_before=["tools"]   # pause before any tool execution
)

config = {"configurable": {"thread_id": "hitl-001"}}

# --- Step 1: Run until the pause point ---
partial = hitl_agent.invoke(
    {"messages": [HumanMessage(content="What is 55 * 88?")]},
    config=config
)

state = hitl_agent.get_state(config)
print("Paused at:", state.next)          # → ('tools',)
print("Pending tool calls:")
last_ai = state.values["messages"][-1]
for tc in getattr(last_ai, "tool_calls", []):
    print(f"  Tool: {tc['name']}, Args: {tc['args']}")

AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
# --- Step 2: Human inspects and decides to continue (or modify state) ---
human_decision = input("Approve tool execution? [y/n]: ")

if human_decision.lower() == "y":
    # Resume by passing None as input — LangGraph continues from the checkpoint
    final = hitl_agent.invoke(None, config=config)
    print("Final answer:", final["messages"][-1].content)
else:
    # Modify state to inject a human override, then resume
    from langchain_core.messages import HumanMessage
    hitl_agent.update_state(
        config,
        {"messages": [HumanMessage(content="Never mind, I'll calculate it myself.")]},
        as_node="agent"   # pretend the update came from agent node
    )
    print("State updated. Tool execution skipped.")

---
## Section 9 — SAP BTP: Generative AI Hub Integration

On SAP BTP, LLMs are accessed via **SAP AI Core → Generative AI Hub**.

### Package name history

| PyPI package | Status | Python imports |
|---|---|---|
| `generative-ai-hub-sdk` | Archived (last: 4.x) | `gen_ai_hub.*` |
| `sap-ai-sdk-gen` | **Current** (6.x, Feb 2026) | `gen_ai_hub.*` (unchanged!) |

The rename was purely cosmetic — **all `gen_ai_hub.*` import paths are identical**.
Drop-in replacement: just change your `pip install` line.

```
LangGraph Agent
      │
      ▼
gen_ai_hub.proxy.langchain.openai.ChatOpenAI
      │
      ▼
SAP AI Core → Generative AI Hub → (GPT-4o, Gemini, Claude, ...)
```

### 9.1 Installation

In [ ]:
# Install SAP SDK + LangGraph (run once in your terminal):
#
#   pip install "sap-ai-sdk-gen[all]" langgraph langchain-core
#
# The [all] extra includes LangChain support and all model providers.

try:
    import gen_ai_hub
    print("SAP Generative AI Hub SDK available (gen_ai_hub module)")
    print(f"  Loaded from: {gen_ai_hub.__file__}")
except ImportError:
    print("Not installed — run: pip install 'sap-ai-sdk-gen[all]'")

### 9.2 Authentication

`sap-ai-sdk-gen` supports two credential formats.

**Format A — Individual env vars** (current default, `sap-ai-sdk-gen` ≥ 5.x):
```bash
export AICORE_CLIENT_ID="sb-..."
export AICORE_CLIENT_SECRET="..."
export AICORE_AUTH_URL="https://<subdomain>.authentication.<region>.hana.ondemand.com"
export AICORE_BASE_URL="https://api.ai.<region>.aws.ml.hana.ondemand.com"
export AICORE_RESOURCE_GROUP="default"
```

**Format B — Single JSON env var** (legacy, still supported):
```bash
export AICORE_SERVICE_KEY='{"clientid":"...","clientsecret":"...","url":"...","serviceurls":{"AI_API_URL":"..."}}'
```

**Format C — Config file** (useful when env vars aren't convenient):
```bash
# Place the service key JSON at:
~/.aicore/config.json
```

All three are read automatically by `get_proxy_client()` — you don't change any Python code, only how you supply credentials.

In [ ]:
import os

# Check which credential format is present
if os.getenv("AICORE_CLIENT_ID"):
    print("Format A detected: individual AICORE_* env vars")
elif os.getenv("AICORE_SERVICE_KEY"):
    print("Format B detected: AICORE_SERVICE_KEY JSON blob")
elif os.path.exists(os.path.expanduser("~/.aicore/config.json")):
    print("Format C detected: ~/.aicore/config.json")
else:
    print("No BTP credentials found — set one of the formats above before running Section 9.")

### 9.3 Initialise the LLM via Generative AI Hub

In [ ]:
# ── Method 1: ChatOpenAI wrapper (most explicit) ───────────────────────────
try:
    from gen_ai_hub.proxy.langchain.openai import ChatOpenAI as GenaihubChatOpenAI
    from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

    proxy_client = get_proxy_client("gen-ai-hub")

    llm_btp = GenaihubChatOpenAI(
        proxy_model_name="gpt-4o",       # model deployed in your AI Core resource group
        proxy_client=proxy_client,
        temperature=0,
        max_tokens=1024,
    )
    print("BTP LLM ready (ChatOpenAI wrapper)")

except ImportError:
    print("generative-ai-hub-sdk not installed — using placeholder.")
    from langchain_openai import ChatOpenAI
    llm_btp = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # fallback to direct OpenAI


# ── Method 2: init_llm helper (shorter, harmonised across providers) ───────
# from gen_ai_hub.proxy.langchain.init_models import init_llm
# llm_btp = init_llm("gpt-4o", max_tokens=1024, temperature=0)


# Quick smoke test
# test_response = llm_btp.invoke("Say hello in one sentence.")
# print(test_response.content)

### 9.4 Full ReAct Agent on SAP BTP

Now swap the LLM for the BTP one and connect tools — everything else is identical.

In [ ]:
from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver


# ── BTP-specific tools (examples) ──────────────────────────────────────────
@tool
def lookup_sales_order(order_id: str) -> str:
    """Look up a SAP sales order by its ID and return order details."""
    # In production: call SAP S/4HANA OData API here
    mock_orders = {
        "SO-1001": "Customer: ACME Corp, Amount: 12,500 EUR, Status: Open",
        "SO-1002": "Customer: SAP SE, Amount: 87,000 EUR, Status: Delivered",
    }
    return mock_orders.get(order_id, f"Order {order_id} not found.")


@tool
def list_open_tickets(priority: str = "all") -> str:
    """List open support tickets. Priority can be 'high', 'medium', 'low', or 'all'."""
    # In production: call SAP Service Cloud or ITSM API
    tickets = [
        {"id": "TKT-001", "priority": "high",   "title": "System outage"},
        {"id": "TKT-002", "priority": "medium", "title": "Performance issue"},
        {"id": "TKT-003", "priority": "low",    "title": "UI display bug"},
    ]
    filtered = [t for t in tickets if priority == "all" or t["priority"] == priority]
    return "\n".join(f"{t['id']} [{t['priority']}]: {t['title']}" for t in filtered)


btp_tools = [lookup_sales_order, list_open_tickets, calculator, get_weather]


# ── Build agent using create_agent ─────────────────────────────────────────
btp_agent = create_agent(
    model=llm_btp,
    tools=btp_tools,
    checkpointer=MemorySaver(),
    system_prompt=(
        "You are a helpful SAP assistant. You help users query SAP data, "
        "answer business questions, and assist with SAP BTP workloads. "
        "Always use tools when they can provide accurate data."
    )
)

print("BTP ReAct agent ready.")

In [ ]:
# ── Run the BTP agent ──────────────────────────────────────────────────────
config = {"configurable": {"thread_id": "btp-session-001"}}

queries = [
    "What are the open high-priority tickets?",
    "Look up sales order SO-1001.",
    "What was the first order I just asked about?",   # tests memory
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"User: {query}")
    response = btp_agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config
    )
    print(f"Agent: {response['messages'][-1].content}")

### 9.5 Streaming Responses

For production UIs on SAP BTP (e.g., integrated into SAP Build Apps or Fiori), stream token-by-token output:

In [ ]:
# Stream: print each token as it arrives
config_stream = {"configurable": {"thread_id": "btp-stream-001"}}

print("Streaming response:")
for chunk in btp_agent.stream(
    {"messages": [HumanMessage(content="What is 42 * 99?")]},
    config=config_stream,
    stream_mode="messages"   # stream individual message tokens
):
    msg, metadata = chunk
    if hasattr(msg, "content") and msg.content:
        print(msg.content, end="", flush=True)
print()

---
## Summary & Next Steps

### What you've learned

| Concept | LangGraph API |
|---------|---------------|
| State | `TypedDict` + `Annotated[..., reducer]` |
| Nodes | Plain Python functions returning partial state dicts |
| Direct edges | `builder.add_edge(A, B)` |
| Conditional edges | `builder.add_conditional_edges(A, router_fn)` |
| Tools | `@tool` decorator + `ToolNode` + `llm.bind_tools()` |
| ReAct loop | Agent → tools → agent (until no tool calls) |
| Memory | `MemorySaver` / `SqliteSaver` + `thread_id` |
| HITL | `interrupt_before=["node"]` + `invoke(None, config)` |
| SAP BTP | `gen_ai_hub.proxy.langchain.openai.ChatOpenAI` |

### Recommended next steps

1. **[LangChain Academy — Intro to LangGraph](https://academy.langchain.com/courses/intro-to-langgraph)** — Free, structured video course
2. **Multi-agent systems** — `langgraph.prebuilt.create_react_agent` as a sub-agent, orchestrator pattern
3. **Long-term memory** — `langgraph-checkpoint-sqlite`, PostgreSQL backend for production
4. **SAP BTP deployment** — Package your agent as a Cloud Foundry app or run inside a CAP service
5. **Observability** — [LangSmith](https://smith.langchain.com) for tracing (free tier available)

### Key references
- [LangGraph GitHub](https://github.com/langchain-ai/langgraph)
- [LangGraph Docs](https://langchain-ai.github.io/langgraph/)
- [SAP Generative AI Hub SDK Docs](https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/gen_ai_hub.html)
- [SAP BTP + LangChain community blog](https://community.sap.com/t5/technology-blog-posts-by-sap/how-to-integrate-sap-ai-core-with-langchain-using-generative-ai-hub-sdk/ba-p/14233467)